# Assignment 2 | Coca-Cola QA System

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import os
from dotenv import load_dotenv, find_dotenv

import glob
from llama_index.core import SimpleDirectoryReader #to Read the PDFs

import warnings
warnings.filterwarnings('ignore')

from llama_index.llms.azure_openai import AzureOpenAI
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding
from llama_index.core import VectorStoreIndex
from llama_index.core.node_parser import SentenceWindowNodeParser
from llama_index.core.postprocessor import MetadataReplacementPostProcessor
from llama_index.core.retrievers import AutoMergingRetriever 
from llama_index.core.node_parser import HierarchicalNodeParser, SentenceSplitter
from llama_index.core.node_parser import get_leaf_nodes, get_root_nodes
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core.storage import StorageContext
from llama_index.core.retrievers.auto_merging_retriever import AutoMergingRetriever
from llama_index.core.response.notebook_utils import display_source_node
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.indices.vector_store.retrievers import VectorIndexAutoRetriever
from llama_index.core.vector_stores.types import MetadataInfo, VectorStoreInfo
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.evaluation import BatchEvalRunner

from llama_index.core.evaluation import (
    FaithfulnessEvaluator,
    RelevancyEvaluator,
    CorrectnessEvaluator,
    RetrieverEvaluator,
    generate_question_context_pairs,
    EmbeddingQAFinetuneDataset
)

from llama_index.core.llama_dataset.generator import RagDatasetGenerator


## Environment Variables

In [ ]:
os.environ["AZURE_OPENAI_API_KEY"] = "YOUR/API/KEY"
os.environ["AZURE_OPENAI_ENDPOINT"] = "YOUR/ENDPOINT"
os.environ["OPENAI_API_VERSION"] = "2024-02-01"

## Load the Data

In [ ]:
directory_path = './data/'
documents = SimpleDirectoryReader(directory_path).load_data()

In [ ]:
len(documents)

In [ ]:
print(documents[0].text)

## LLM Configuration

In [ ]:
llm = AzureOpenAI(
    engine="gpt-4o-mini",
    model="gpt-4o-mini",
    temperature=0.0,
)

## Defining the Embedding Model

In [ ]:
embed_model = AzureOpenAIEmbedding(
    model="text-embedding-3-small"
)

## Defining the Node Parser

In [ ]:
window_node_parser = SentenceWindowNodeParser.from_defaults(
    window_size=3,
    window_metadata_key="window",
    original_text_metadata_key="original_text",
)

In [ ]:
#Extracting the Nodes
window_nodes = window_node_parser.get_nodes_from_documents(documents)

In [ ]:
#Building the Indeces
sentence_index = VectorStoreIndex(window_nodes, embed_model=embed_model, show_progress=True)

## Querying

### Sentence Window Retriever

### MetadataReplacementPostProcessor
It replaces the actual sentence in each node with it's surrounding context.

In [ ]:
sentence_query_engine = sentence_index.as_query_engine(
    llm=llm,
    similarity_top_k=2,
    node_postprocessors=[
        MetadataReplacementPostProcessor(target_metadata_key="window")
        #window here is going to replace the original text by the surrouding only at the time of query
    ],
)

In [ ]:
sentence_window_response = sentence_query_engine.query(
    "What is the main ingredient in all products?"
)
print(sentence_window_response)

In [ ]:
sentence = sentence_window_response.source_nodes[0].node.metadata["original_text"]
print(sentence)

### Auto Merging Retriever

looks at a set of leaf nodes and recursively "merges" subsets of leaf nodes that reference a parent node beyond a given threshold. This allows us to consolidate potentially disparate, smaller contexts into a larger context that might help synthesis.

You can define this hierarchy yourself over a set of documents, or you can make use of our brand-new text parser: a HierarchicalNodeParser that takes in a candidate set of documents and outputs an entire hierarchy of nodes, from "coarse-to-fine".

By default, the hierarchy is:
- 1st level: chunk size 2048
- 2nd level: chunk size 512
- 3rd level: chunk size 128

The leaf nodes are indexed and retrieved via a vector store - they will first be directly retrieved via similarity search.
The other nodes will be retrieved from a docstore.

In [ ]:
automerge_node_parser = HierarchicalNodeParser.from_defaults()

In [ ]:
automerge_nodes = automerge_node_parser.get_nodes_from_documents(documents)
len(automerge_nodes)

In [ ]:
leaf_nodes = get_leaf_nodes(automerge_nodes)
len(leaf_nodes)

In [ ]:
root_nodes = get_root_nodes(automerge_nodes)
len(root_nodes)

#### Load into Storage

We define a docstore, which we load all nodes into.

We then define a `VectorStoreIndex` containing just the leaf-level nodes.

In [ ]:
docstore = SimpleDocumentStore()

# insert nodes into docstore
docstore.add_documents(automerge_nodes)

# define storage context (will include vector store by default too)
storage_context = StorageContext.from_defaults(docstore=docstore)

In [ ]:
base_index = VectorStoreIndex(
    leaf_nodes,
    embed_model = embed_model,
    storage_context=storage_context,
)

In [ ]:
#Define the retriever
base_retriever = base_index.as_retriever(similarity_top_k=6)
retriever = AutoMergingRetriever(base_retriever, storage_context, verbose=True)

In [ ]:
query_str = (
    "What is the main ingredient in all products?"
)

base_nodes = base_retriever.retrieve(query_str)
nodes = retriever.retrieve(query_str)

In [ ]:
len(nodes)

In [ ]:
for node in nodes:
    display_source_node(node, source_length=10000)

In [ ]:
for node in base_nodes:
    display_source_node(node, source_length=10000)

##### Plug into Query Engine

In [ ]:
query_engine = RetrieverQueryEngine.from_args(retriever, llm=llm)
base_query_engine = RetrieverQueryEngine.from_args(base_retriever, llm=llm)

In [ ]:
response = query_engine.query(query_str)
base_response = base_query_engine.query(query_str)

In [ ]:
print(str(response))
print(str(base_response))

### Sub Question

In [ ]:
query_engine_tools = [
    QueryEngineTool(
        query_engine=query_engine,
        metadata=ToolMetadata(name='KO_10k', description='Provides information about Cocacola financials for year 2016 to 2026')
    ),
]

s_engine = SubQuestionQueryEngine.from_defaults(query_engine_tools=query_engine_tools, llm=llm)


response = await s_engine.aquery('Compare and contrast the customer segments and geographies that grew the fastest')


In [ ]:
print(response.response)

## Evaluation

In [ ]:
llm_judge = AzureOpenAI(
    engine="gpt-4o",
    model="gpt-4o",
    temperature=0.0,
)

data_generator = RagDatasetGenerator.from_documents(
    documents,
    llm=llm_judge,
    num_questions_per_chunk=1
)

In [ ]:
eval_dataset = data_generator.generate_dataset_from_nodes()

eval_questions = [example.query for example in eval_dataset.examples]
eval_answers = [example.reference_answer for example in eval_dataset.examples]

In [ ]:
# Query Engine
query_engine = sentence_index.as_query_engine(llm=llm)
# Create Evaluators
relevancy_evaluator = RelevancyEvaluator(llm=llm)
faithfulness_evaluator = FaithfulnessEvaluator(llm=llm_judge)
correctness_evaluator = CorrectnessEvaluator(llm=llm_judge)

In [ ]:
runner = BatchEvalRunner(
    {
     "faithfulness": faithfulness_evaluator,
     "relevancy": relevancy_evaluator,
     "correctness": correctness_evaluator
     },
    workers=8,
)

eval_results = await runner.aevaluate_queries(
    query_engine, queries=eval_questions, reference = eval_answers
)

In [ ]:
def get_eval_results(key, eval_results):
    results = eval_results[key]
    correct = 0
    for result in results:
        if result.passing:
            correct += 1
    score = correct / len(results)
    print(f"{key} Score: {score}")
    return score

In [ ]:
get_eval_results("faithfulness", eval_results)
get_eval_results("relevancy", eval_results)
_ = get_eval_results("correctness", eval_results)